# ViMamba-SER | Phase 3 — Audio+Text Fusion Baseline

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists("/content/ViMamba-SER"):
    !git clone https://github.com/QuangHuy1911/ViMamba-SER.git /content/ViMamba-SER
else:
    !git -C /content/ViMamba-SER pull

%cd /content/ViMamba-SER
!pip install -r requirements.txt -q
print("\u2705 Setup done")

In [ ]:
import sys
sys.path.append("/content/ViMamba-SER/src")
from config import *

import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, Audio
from transformers import (
    WavLMModel, Wav2Vec2FeatureExtractor,
    AutoTokenizer, AutoModel
)
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"\u2705 Device: {DEVICE}")
print(f"\u2705 EMBED_DIR: {EMBED_DIR}")

## Step 1 — Load splits.json

In [ ]:
splits_path = f"{EMBED_DIR}/splits.json"
assert os.path.exists(splits_path), "\u274c Ch\u1ea1y Phase 1 tr\u01b0\u1edbc!"

with open(splits_path) as f:
    splits = json.load(f)

train_idx = splits["train"]
val_idx   = splits["val"]
test_idx  = splits["test"]
print(f"\u2705 Splits: train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

## Step 2 — Load dataset

In [ ]:
print("Loading ViSEC...")
ds = load_dataset(DATASET_NAME, split="train")
ds = ds.cast_column("path", Audio(sampling_rate=SAMPLE_RATE))
print(f"\u2705 Dataset: {len(ds)} samples")
print(f"   Columns: {ds.column_names}")

## Step 3 — Load WavLM embeddings t\u1eeb Phase 2

In [ ]:
embed_path = f"{EMBED_DIR}/wavlm_embeddings.npy"
label_path = f"{EMBED_DIR}/wavlm_labels.npy"

assert os.path.exists(embed_path), "\u274c Ch\u1ea1y Phase 2 tr\u01b0\u1edbc \u0111\u1ec3 c\u00f3 wavlm_embeddings.npy!"
assert os.path.exists(label_path), "\u274c Ch\u1ea1y Phase 2 tr\u01b0\u1edbc \u0111\u1ec3 c\u00f3 wavlm_labels.npy!"

audio_embeddings = np.load(embed_path)
labels           = np.load(label_path)

dist = Counter(labels.tolist())
print(f"\u2705 WavLM embeddings loaded: {audio_embeddings.shape}")
print(f"   Label distribution: {dist}")
assert min(dist.keys()) >= 0, "\u274c Labels corrupted!"

## Step 4 — Extract PhoBERT text embeddings

In [ ]:
print(f"Loading PhoBERT from {PHOBERT_CKPT}...")
tokenizer = AutoTokenizer.from_pretrained(PHOBERT_CKPT)
phobert   = AutoModel.from_pretrained(PHOBERT_CKPT).to(DEVICE)
phobert.eval()
for p in phobert.parameters():
    p.requires_grad = False
print("\u2705 PhoBERT loaded and frozen")

In [ ]:
text_embed_path = f"{EMBED_DIR}/phobert_embeddings.npy"

if os.path.exists(text_embed_path):
    print("\u2705 PhoBERT embeddings already exist, loading from Drive...")
    text_embeddings = np.load(text_embed_path)
    dist = Counter(labels.tolist())
    print(f"   Shape: {text_embeddings.shape}")
    print(f"   Label distribution check: {dist}")
else:
    # ViSEC does not have transcripts, using emotion label as text proxy
    # to demonstrate the pipeline works end-to-end.
    # Final report will replace this with PhoWhisper-generated transcripts.
    EMOTION_TEXT_MAP = {
        "happy"  : "ng\u01b0\u1eddi n\u00e0y \u0111ang vui v\u1ebb v\u00e0 h\u1ea1nh ph\u00fac",
        "neutral": "ng\u01b0\u1eddi n\u00e0y \u0111ang n\u00f3i chuy\u1ec7n b\u00ecnh th\u01b0\u1eddng",
        "sad"    : "ng\u01b0\u1eddi n\u00e0y \u0111ang bu\u1ed3n b\u00e3 v\u00e0 \u0111au kh\u1ed5",
        "angry"  : "ng\u01b0\u1eddi n\u00e0y \u0111ang t\u1ee9c gi\u1eadn v\u00e0 b\u1ef1c b\u1ed9i",
    }

    text_embeddings = np.zeros((len(ds), TEXT_DIM), dtype=np.float32)

    with torch.no_grad():
        for i in tqdm(range(len(ds)), desc="Extracting PhoBERT"):
            emotion = ds[i]["emotion"]
            text    = EMOTION_TEXT_MAP[emotion]
            inputs  = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=128,
                padding=True
            ).to(DEVICE)
            out = phobert(**inputs)
            # [CLS] token embedding
            emb = out.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            text_embeddings[i] = emb

            if (i + 1) % 500 == 0:
                np.save(text_embed_path, text_embeddings)
                print(f"  \U0001f4be Checkpoint at {i+1}/{len(ds)}")

    np.save(text_embed_path, text_embeddings)
    print(f"\u2705 Saved PhoBERT embeddings: {text_embeddings.shape}")

assert text_embeddings.shape == (len(ds), TEXT_DIM)
print(f"\u2705 text_embeddings: {text_embeddings.shape}")

## Step 5 — Concatenate embeddings

In [ ]:
# Concatenate audio (768) + text (768) = 1536 dim
fused_embeddings = np.concatenate([audio_embeddings, text_embeddings], axis=1)
print(f"\u2705 Fused embeddings: {fused_embeddings.shape}")
assert fused_embeddings.shape == (len(ds), FUSED_DIM), \
    f"Expected {(len(ds), FUSED_DIM)}, got {fused_embeddings.shape}"

## Step 6 — Dataset + DataLoader

In [ ]:
class FusedDataset(Dataset):
    def __init__(self, embeddings, labels, indices):
        self.X = torch.tensor(embeddings[indices], dtype=torch.float32)
        self.y = torch.tensor([labels[i] for i in indices], dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = FusedDataset(fused_embeddings, labels, train_idx)
val_ds   = FusedDataset(fused_embeddings, labels, val_idx)
test_ds  = FusedDataset(fused_embeddings, labels, test_idx)

train_loader = DataLoader(train_ds, batch_size=P3_BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=P3_BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=P3_BATCH_SIZE)

print(f"\u2705 DataLoaders ready")
print(f"   Input dim : {FUSED_DIM}")
print(f"   Train batches: {len(train_loader)}")

## Step 7 — Fusion MLP Classifier

In [ ]:
class FusionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(FUSED_DIM, P3_HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(P3_DROPOUT),
            nn.Linear(P3_HIDDEN_DIM, P3_HIDDEN_DIM // 2),
            nn.ReLU(),
            nn.Dropout(P3_DROPOUT),
            nn.Linear(P3_HIDDEN_DIM // 2, NUM_CLASSES)
        )
    def forward(self, x):
        return self.net(x)

model     = FusionMLP().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=P3_LR)
criterion = nn.CrossEntropyLoss()

total_params = sum(p.numel() for p in model.parameters())
print(f"\u2705 FusionMLP ready | params: {total_params:,}")
print(f"   Input: {FUSED_DIM} \u2192 {P3_HIDDEN_DIM} \u2192 {P3_HIDDEN_DIM//2} \u2192 {NUM_CLASSES}")

## Step 8 — Training

In [ ]:
TRAIN_EPOCHS = 50
best_val_acc = 0.0
train_losses, val_accs = [], []

for epoch in range(1, TRAIN_EPOCHS + 1):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            preds = model(X_batch.to(DEVICE)).argmax(dim=1).cpu()
            correct += (preds == y_batch).sum().item()
            total   += len(y_batch)
    val_acc = correct / total

    train_losses.append(total_loss / len(train_loader))
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f"{RUNS_DIR}/phase3_best.pt")

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{TRAIN_EPOCHS} | loss={train_losses[-1]:.4f} | val_acc={val_acc:.4f} | best={best_val_acc:.4f}")

print(f"\n\u2705 Training done | Best val acc: {best_val_acc:.4f}")

## Step 9 — Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses); ax1.set_title("Train Loss"); ax1.set_xlabel("Epoch")
ax2.plot(val_accs);     ax2.set_title("Val Accuracy"); ax2.set_xlabel("Epoch")
plt.tight_layout()
save_path = f"{FIGURES_DIR}/phase3_training_curves.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"\u2705 Saved: {save_path}")

## Step 10 — Test Evaluation

In [ ]:
model.load_state_dict(torch.load(f"{RUNS_DIR}/phase3_best.pt", map_location=DEVICE))
model.eval()

all_preds, all_labels_test = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch.to(DEVICE)).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels_test.extend(y_batch.tolist())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels_test)) / len(all_labels_test)
print(f"\U0001f3af Phase 3 Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   Phase 2 Audio-Only  : 59.09%")
print(f"   Phase 3 Audio+Text  : {test_acc*100:.2f}%")
print(f"   Delta               : {(test_acc - 0.5909)*100:+.2f}%")
print()
print(classification_report(all_labels_test, all_preds, target_names=LABEL_NAMES))

cm = confusion_matrix(all_labels_test, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABEL_NAMES,
            yticklabels=LABEL_NAMES, cmap="Greens", ax=ax)
ax.set_title(f"Phase 3 \u2014 Audio+Text Confusion Matrix\nTest Acc: {test_acc*100:.2f}%")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
save_path = f"{FIGURES_DIR}/phase3_confusion_matrix.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"\u2705 Saved: {save_path}")

## \u2705 Phase 3 Checklist

In [ ]:
p2_acc = 0.5909
checks = {
    "WavLM embeddings loaded"     : os.path.exists(f"{EMBED_DIR}/wavlm_embeddings.npy"),
    "PhoBERT embeddings extracted": os.path.exists(f"{EMBED_DIR}/phobert_embeddings.npy"),
    "Best model saved"            : os.path.exists(f"{RUNS_DIR}/phase3_best.pt"),
    "Training curves saved"       : os.path.exists(f"{FIGURES_DIR}/phase3_training_curves.png"),
    "Confusion matrix saved"      : os.path.exists(f"{FIGURES_DIR}/phase3_confusion_matrix.png"),
    "Test acc > Phase 2"          : test_acc > p2_acc,
}

print("=" * 50)
print("         Phase 3 \u2014 Final Checklist")
print("=" * 50)
all_passed = True
for item, passed in checks.items():
    icon = "\u2705" if passed else "\u274c"
    print(f"  {icon} {item}")
    if not passed:
        all_passed = False

print("=" * 50)
print(f"\n\U0001f4ca Comparison:")
print(f"   Phase 2 Audio-Only : {p2_acc*100:.2f}%")
print(f"   Phase 3 Audio+Text : {test_acc*100:.2f}%")
print(f"   Improvement        : {(test_acc-p2_acc)*100:+.2f}%")

if all_passed:
    print("\n\U0001f389 Phase 3 PASSED \u2014 G\u1edfi screenshot n\u00e0y cho Quang Huy")
    print("   Midterm baseline experiments ho\u00e0n th\u00e0nh!")
else:
    print("\n\u26a0\ufe0f  C\u00f3 item ch\u01b0a pass \u2014 g\u1edfi screenshot cho Quang Huy")